# Piloto de predicción horaria con torres GRU-D

Este notebook reutiliza los módulos del proyecto. La configuración inicial toma **300 pacientes en total del train de A**, conserva todas sus horas y reserva el 20 % de los pacientes para validación. La prueba final está desactivada por defecto.

Antes de empezar, ejecuta `uv sync --extra model --group dev` en la raíz del proyecto, selecciona el kernel de `.venv` y comprueba que `cleaning.ipynb` haya exportado las particiones.


In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from tensorflow import keras

from appr_project.data import (
    PatientHours,
    Standardization,
    WindowBatches,
    load_partition,
    split_patients,
)
from appr_project.evaluation import predict_hours, summarize_metrics
from appr_project.training import Hyperparameters, train_experiment

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
DATA_DIR = ROOT / "data" / "processed"


## 1. Configuración del piloto

`source` admite `"A"`, `"B"` y `"A+B"`. Para usar todos los pacientes de entrenamiento, cambia `sample_patients` a `None`. Con `folds=None` se usa una sola partición; un entero como `5` activa la validación cruzada.

`trials=1` entrena solo la configuración base. Con `trials=4` se prueban esa configuración y tres alternativas de la búsqueda aleatoria. El costo de la búsqueda aumenta con los ensayos y los folds.


In [ ]:
settings = dict(
    source="A",
    sample_patients=2000,
    validation_fraction=0.20,
    folds=None,
    trials=10,
    epochs=20,
    patience=4,
    seed=2026,
)

base = Hyperparameters(
    window=15,
    units=8,
    dense_units=32,
    dropout=0.20,
    learning_rate=0.001,
    batch_size=128,
)


## 2. Inspeccionar la partición y un batch

Se muestran los pacientes, sus horas y cuántos tienen alguna etiqueta positiva. En CV, la vista previa corresponde al primer fold. La muestra y la partición se reproducen al entrenar con la misma semilla.

Cada entrada temporal tiene dimensiones `[batch, horas, variables, 4]`. Los cuatro canales son el valor estandarizado, la máscara de medición, el tiempo desde la medición previa y su valor. La máscara `valid_hours` marca las horas reales de la ventana.


In [8]:
frame = load_partition(DATA_DIR, "train", settings["source"])

train_ids, val_ids = next(split_patients(
    frame,
    validation_fraction=settings["validation_fraction"],
    folds=settings["folds"],
    seed=settings["seed"],
    sample_patients=settings["sample_patients"],
))

train_preview = frame.loc[frame.patient_id.isin(train_ids)]
val_preview = frame.loc[frame.patient_id.isin(val_ids)]

summary = []
for name, partition in (("train", train_preview), ("validation", val_preview)):
    patient_labels = partition.groupby("patient_id").SepsisLabel.max()
    summary.append({
        "partición": name,
        "pacientes": len(patient_labels),
        "pacientes positivos": int(patient_labels.sum()),
        "horas": len(partition),
    })
display(pd.DataFrame(summary).set_index("partición"))


,pacientes,pacientes positivos,horas
partición,,,
train,1600,141,62270
validation,400,35,15554


In [9]:
scaler = Standardization.fit(train_preview)
first_patient = train_preview.loc[train_preview.patient_id.eq(train_ids[0])]
hours = PatientHours(first_patient, scaler)
batches = WindowBatches(hours, base.window, base.batch_size)
inputs, labels = batches[0]

print("Formas de entrada:", {name: array.shape for name, array in inputs.items()})
print("Forma de las etiquetas:", labels.shape)
print("Paciente:", hours.metadata["patient_id"].iloc[0])
print("Horas válidas de la primera ventana:", inputs["valid_hours"][0].astype(int).tolist())
print("Etiqueta de la primera ventana:", int(labels[0, 0]))


Formas de entrada: {'temporal': (11, 15, 26, 4), 'valid_hours': (11, 15), 'static': (11, 10)}
Forma de las etiquetas: (11, 1)
Paciente: A_015805
Horas válidas de la primera ventana: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Etiqueta de la primera ventana: 0


## 3. Entrenar y revisar los resultados

Cada ejecución de la celda crea una carpeta nueva en `runs/` con el modelo `best.keras`, la configuración, los tiempos y las métricas. Las métricas de validación sirven para elegir el modelo y el umbral. Con una muestra de 300 pacientes, el objetivo del piloto es verificar el flujo y estimar el costo.


In [10]:
run_dir = ROOT / "runs" / f"notebook-{settings['source']}-{datetime.now():%Y%m%d-%H%M%S-%f}"
validation_metrics = train_experiment(frame, run_dir, base=base, **settings)

print(f"Resultados: {run_dir}")
display(validation_metrics.round(4))


Trial 1/10, fold 1/1: Hyperparameters(window=15, units=8, dense_units=32, dropout=0.2, learning_rate=0.001, batch_size=128)


c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/20
487/487 - 51s - 105ms/step - loss: 0.1315 - pr_auc: 0.0257 - val_loss: 0.1014 - val_pr_auc: 0.0412
Epoch 2/20
487/487 - 41s - 85ms/step - loss: 0.0977 - pr_auc: 0.0736 - val_loss: 0.0998 - val_pr_auc: 0.0476
Epoch 3/20
487/487 - 40s - 83ms/step - loss: 0.0920 - pr_auc: 0.1219 - val_loss: 0.0972 - val_pr_auc: 0.0592
Epoch 4/20
487/487 - 49s - 100ms/step - loss: 0.0897 - pr_auc: 0.1489 - val_loss: 0.0961 - val_pr_auc: 0.0692
Epoch 5/20
487/487 - 81s - 167ms/step - loss: 0.0875 - pr_auc: 0.1752 - val_loss: 0.0966 - val_pr_auc: 0.0724
Epoch 6/20
487/487 - 59s - 122ms/step - loss: 0.0854 - pr_auc: 0.2034 - val_loss: 0.0943 - val_pr_auc: 0.0818
Epoch 7/20
487/487 - 78s - 161ms/step - loss: 0.0828 - pr_auc: 0.2174 - val_loss: 0.0927 - val_pr_auc: 0.0889
Epoch 8/20
487/487 - 56s - 115ms/step - loss: 0.0801 - pr_auc: 0.2465 - val_loss: 0.0928 - val_pr_auc: 0.0936
Epoch 9/20
487/487 - 56s - 115ms/step - loss: 0.0786 - pr_auc: 0.2631 - val_loss: 0.0929 - val_pr_auc: 0.0999
Epoch 10/20


c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/20
487/487 - 102s - 209ms/step - loss: 0.1625 - pr_auc: 0.0227 - val_loss: 0.1033 - val_pr_auc: 0.0359
Epoch 2/20
487/487 - 140s - 288ms/step - loss: 0.1035 - pr_auc: 0.0404 - val_loss: 0.1023 - val_pr_auc: 0.0430
Epoch 3/20
487/487 - 71s - 146ms/step - loss: 0.0993 - pr_auc: 0.0572 - val_loss: 0.1011 - val_pr_auc: 0.0470
Epoch 4/20
487/487 - 73s - 150ms/step - loss: 0.0972 - pr_auc: 0.0765 - val_loss: 0.0999 - val_pr_auc: 0.0543
Epoch 5/20
487/487 - 69s - 141ms/step - loss: 0.0944 - pr_auc: 0.0931 - val_loss: 0.0990 - val_pr_auc: 0.0570
Epoch 6/20
487/487 - 69s - 142ms/step - loss: 0.0924 - pr_auc: 0.1178 - val_loss: 0.0971 - val_pr_auc: 0.0667
Epoch 7/20
487/487 - 74s - 152ms/step - loss: 0.0907 - pr_auc: 0.1302 - val_loss: 0.0966 - val_pr_auc: 0.0697
Epoch 8/20
487/487 - 78s - 160ms/step - loss: 0.0893 - pr_auc: 0.1456 - val_loss: 0.0965 - val_pr_auc: 0.0741
Epoch 9/20
487/487 - 76s - 156ms/step - loss: 0.0879 - pr_auc: 0.1688 - val_loss: 0.0966 - val_pr_auc: 0.0776
Epoch 10

c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


... (12 outputs left out)

487/487 - 41s - 84ms/step - loss: 0.1665 - pr_auc: 0.0211 - val_loss: 0.1069 - val_pr_auc: 0.0228
Epoch 2/20
487/487 - 34s - 70ms/step - loss: 0.1067 - pr_auc: 0.0302 - val_loss: 0.1038 - val_pr_auc: 0.0368
Epoch 3/20
487/487 - 34s - 70ms/step - loss: 0.1018 - pr_auc: 0.0467 - val_loss: 0.1028 - val_pr_auc: 0.0404
Epoch 4/20
487/487 - 33s - 68ms/step - loss: 0.0992 - pr_auc: 0.0634 - val_loss: 0.1013 - val_pr_auc: 0.0464
Epoch 5/20
487/487 - 32s - 65ms/step - loss: 0.0958 - pr_auc: 0.0825 - val_loss: 0.1000 - val_pr_auc: 0.0511
Epoch 6/20
487/487 - 34s - 69ms/step - loss: 0.0935 - pr_auc: 0.1050 - val_loss: 0.0978 - val_pr_auc: 0.0614
Epoch 7/20
487/487 - 32s - 67ms/step - loss: 0.0917 - pr_auc: 0.1174 - val_loss: 0.0969 - val_pr_auc: 0.0662
Epoch 8/20
487/487 - 36s - 73ms/step - loss: 0.0901 - pr_auc: 0.1388 - val_loss: 0.0967 - val_pr_auc: 0.0712
Epoch 9/20
487/487 - 33s - 69ms/step - loss: 0.0885 - pr_auc: 0.1613 - val_loss: 0.0962 - val_pr_auc: 0.0770
Epoch 10/20
487/487 - 34s - 71

c:\Users\palom\Documents\College\APPR\APPR_project\.venv\Lib\site-packages\keras\src\layers\layer.py:1039: UserWarning: Layer 'reshape_1' (of type Reshape) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


487/487 - 41s - 84ms/step - loss: 0.1863 - pr_auc: 0.0208 - val_loss: 0.1073 - val_pr_auc: 0.0208
Epoch 2/20
487/487 - 35s - 72ms/step - loss: 0.1041 - pr_auc: 0.0319 - val_loss: 0.1032 - val_pr_auc: 0.0339
Epoch 3/20
487/487 - 37s - 75ms/step - loss: 0.1002 - pr_auc: 0.0476 - val_loss: 0.1014 - val_pr_auc: 0.0437
Epoch 4/20
487/487 - 37s - 77ms/step - loss: 0.0978 - pr_auc: 0.0602 - val_loss: 0.1004 - val_pr_auc: 0.0482
Epoch 5/20
487/487 - 37s - 76ms/step - loss: 0.0945 - pr_auc: 0.0848 - val_loss: 0.0996 - val_pr_auc: 0.0465
Epoch 6/20
487/487 - 36s - 74ms/step - loss: 0.0923 - pr_auc: 0.1127 - val_loss: 0.0980 - val_pr_auc: 0.0527
Epoch 7/20
487/487 - 35s - 73ms/step - loss: 0.0902 - pr_auc: 0.1294 - val_loss: 0.0970 - val_pr_auc: 0.0578
Epoch 8/20
487/487 - 41s - 84ms/step - loss: 0.0886 - pr_auc: 0.1473 - val_loss: 0.0970 - val_pr_auc: 0.0619
Epoch 9/20
487/487 - 36s - 73ms/step - loss: 0.0873 - pr_auc: 0.1670 - val_loss: 0.0963 - val_pr_auc: 0.0660
Epoch 10/20
487/487 - 36s - 75

,source,patients,hours,average_precision,auroc,utility,alert_fraction,threshold
0,all,400,15554,0.1128,0.7983,0.3819,0.3337,0.0064
1,A,400,15554,0.1128,0.7983,0.3819,0.3337,0.0064


In [11]:
display(pd.read_csv(run_dir / "trials.csv").round(4))

fold_metrics = pd.read_csv(run_dir / "fold_metrics.csv")
timing_columns = [
    "trial", "fold", "train_windows", "validation_windows",
    "preparation_seconds", "epochs_run", "seconds_per_epoch",
]
display(fold_metrics.loc[fold_metrics.source.eq("all"), timing_columns].round(2))


,trial,window,units,dense_units,dropout,learning_rate,batch_size,mean_validation_ap,std_validation_ap
0,1,15,8,32,0.2,0.0010,128,0.1115,0.0
1,2,8,32,32,0.2,0.0003,128,0.1059,0.0
2,3,15,32,64,0.1,0.0010,256,0.0886,0.0
3,4,8,8,64,0.2,0.0003,128,0.1087,0.0
4,5,24,32,64,0.2,0.0010,64,0.0904,0.0
5,6,15,16,32,0.2,0.0010,64,0.1128,0.0
6,7,15,16,32,0.1,0.0003,128,0.0902,0.0
7,8,15,8,64,0.1,0.0010,128,0.0956,0.0
8,9,8,16,32,0.2,0.0003,128,0.1026,0.0
9,10,15,8,32,0.1,0.0003,128,0.0867,0.0


,trial,fold,train_windows,validation_windows,preparation_seconds,epochs_run,seconds_per_epoch
0,1,1,62270,15554,1.92,17,52.25
2,2,1,62270,15554,1.73,20,77.12
4,3,1,62270,15554,0.76,20,115.40
6,4,1,62270,15554,0.63,20,20.71
8,5,1,62270,15554,0.74,17,173.03
10,6,1,62270,15554,0.84,14,61.89
12,7,1,62270,15554,0.82,20,54.91
14,8,1,62270,15554,0.67,16,34.99
16,9,1,62270,15554,0.80,20,34.32
18,10,1,62270,15554,0.57,20,39.02


## 4. Evaluación final opcional

Activa esta celda cuando hayas elegido la configuración definitiva. La utilidad y la proporción de horas con alerta en test se calculan con el umbral guardado en la corrida. Los resultados se muestran y se guardan por origen y de forma agregada. Usa `positive_hour_fraction` como referencia para el AP.


In [7]:
evaluate_test = False
test_source = "A+B"  # También admite "A" o "B".

if evaluate_test:
    config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
    saved_model = keras.models.load_model(run_dir / "best.keras", compile=False)
    saved_scaler = Standardization(**config["standardization"])
    test = load_partition(DATA_DIR, "test", test_source)
    test_batches = WindowBatches(
        PatientHours(test, saved_scaler),
        config["hyperparameters"]["window"],
        config["hyperparameters"].get("batch_size", config.get("batch_size")),
    )
    test_predictions = predict_hours(saved_model, test_batches)
    test_metrics = summarize_metrics(test_predictions, config["threshold"])
    test_predictions.to_csv(run_dir / f"test_{test_source}_predictions.csv", index=False)
    test_metrics.to_csv(run_dir / f"test_{test_source}_metrics.csv", index=False)
    display(test_metrics.round(4))
